# イラストAI Tagger：Colab API

Docker不要。PixAI v1.0 FP16のAPIを起動し、Eagle Clientから接続します。

1. 「ランタイム」→「ランタイムのタイプを変更」でGPUを選びます。
2. Colab Pro／計算ユニット残高を確認し、「ランタイム」→「すべてのセルを実行」。
3. 最後に出る **接続先URL・APIトークン**を、Clientの「接続設定」へ貼り付けます。
4. 作業終了時は「ランタイム」→「接続を解除してランタイムを削除」。

Clientも **feature/colab-api** ブランチ版が必要です。main版はHTTPSに未対応です。

初回は約1GBのモデルを取得します。Colabの実行環境・GPUの割り当てにより起動時間は変わります。
このノートブックは画像・動画のタグ付け用です。説明文生成用のQwenは導入しません。
トークンは実行時に生成します。実行結果を含むノートブックは共有しないでください。
外部トンネルはCloudflare Quick Tunnel（公式配布）を使用します。Colabの共通利用制約は別途適用されます。


In [ ]:
import os, sys, subprocess, platform, time, re, secrets, socket, io
from pathlib import Path

REPO_URL = "https://github.com/sabiaka/2DIlust_Auto_Ai_tagging_APIServer.git"
REPO_REF = "feature/colab-api"
REPO_DIR = Path("/content/illustration-tagger-colab")

if not Path("/content").exists() or platform.system() != "Linux":
    raise RuntimeError("このノートブックはLinuxのColab GPUランタイムで実行してください。")
subprocess.run(["nvidia-smi"], check=True)
import torch
if not torch.cuda.is_available():
    raise RuntimeError("GPUが使えません。ランタイムの種類をGPUに変更してください。")
print("GPU:", torch.cuda.get_device_name(0))
if not REPO_DIR.exists():
    subprocess.run(["git", "clone", "--branch", REPO_REF, "--single-branch", REPO_URL, str(REPO_DIR)], check=True)
elif not (REPO_DIR / "api_auth.py").exists():
    raise RuntimeError("既存フォルダがColab対応版ではありません。別フォルダまたは新規ランタイムで実行してください。")
subprocess.run([sys.executable, "-m", "pip", "install", "-r", str(REPO_DIR / "requirements-colab.txt")], check=True)
print("準備完了")

## API起動とGPU推論の確認
GPU・推論・認証を確認してから外部接続を開始します。

In [ ]:
import requests
from PIL import Image

# Re-running this cell stops only processes started by this notebook.
def stop_services():
    for name in ("tunnel_process", "api_process"):
        process = globals().get(name)
        if process is not None and process.poll() is None:
            process.terminate()
            try:
                process.wait(timeout=10)
            except subprocess.TimeoutExpired:
                process.kill()
                process.wait(timeout=10)

stop_services()
with socket.socket() as probe:
    if probe.connect_ex(("127.0.0.1", 8000)) == 0:
        raise RuntimeError("ポート8000が使用中です。前のAPIを停止してください。")
API_TOKEN = secrets.token_urlsafe(32)
headers = {"Authorization": "Bearer " + API_TOKEN}
env = os.environ.copy()
env.update(API_AUTH_TOKEN=API_TOKEN, TAGGER_BACKEND="pixai", PIXAI_MODEL_NAME="v1.0",
           TAGGER_WARMUP="0", ORT_PRELOAD_NVIDIA="1", HF_HOME="/content/tagger-model-cache", PYTHONUNBUFFERED="1")
# Do not reuse a model path inherited from another environment.
env.pop("PIXAI_MODEL_PATH", None)
log_path = Path("/content/tagger-api.log")
api_log = log_path.open("w")
api_process = subprocess.Popen([sys.executable, "-u", "run_server.py", "--host", "127.0.0.1", "--port", "8000"],
                               cwd=REPO_DIR, env=env, stdout=api_log, stderr=subprocess.STDOUT)
try:
    deadline = time.monotonic() + 600
    while True:
        if api_process.poll() is not None:
            raise RuntimeError("APIの起動に失敗しました。")
        try:
            response = requests.get("http://127.0.0.1:8000/tagger-info", headers=headers, timeout=5)
            if response.status_code == 200:
                info = response.json()
                break
        except requests.RequestException:
            pass
        if time.monotonic() > deadline:
            raise TimeoutError("APIの起動待ちが10分を超えました。")
        time.sleep(1)
    if "CUDAExecutionProvider" not in info.get("providers", []):
        raise RuntimeError("GPU推論が有効ではありません。CPUへの自動切替は行いません。")
    if requests.get("http://127.0.0.1:8000/tagger-info", timeout=10).status_code != 401:
        raise RuntimeError("API認証が有効ではありません。")
    sample = io.BytesIO()
    Image.new("RGB", (256, 256), "white").save(sample, format="PNG")
    started = time.monotonic()
    result = requests.post("http://127.0.0.1:8000/tag", headers=headers,
                           files={"file": ("test.png", sample.getvalue(), "image/png")}, timeout=120)
    result.raise_for_status()
    assert isinstance(result.json().get("tags"), list)
    print("モデル:", info["pixai_model_name"], " / GPU:", info["providers"])
    print("テスト画像の推論成功:", round(time.monotonic() - started, 2), "秒")
except Exception:
    stop_services()
    print(log_path.read_text(errors="replace")[-6000:])
    raise

## 一時HTTPS URLを作成
URLを知るだけではAPIを使えません。Clientには下で表示されるトークンも設定します。
URLとトークンは再起動時に変わります。

In [ ]:
binary = Path("/content/cloudflared")
if not binary.exists():
    architecture = {"x86_64": "amd64", "aarch64": "arm64"}.get(platform.machine())
    if not architecture:
        raise RuntimeError("未対応のCPUアーキテクチャです。")
    download = requests.get("https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-" + architecture, timeout=120)
    download.raise_for_status()
    binary.write_bytes(download.content)
    binary.chmod(0o700)
subprocess.run([str(binary), "--version"], check=True)
tunnel_path = Path("/content/tagger-tunnel.log")
tunnel_log = tunnel_path.open("w")
tunnel_process = subprocess.Popen([str(binary), "tunnel", "--url", "http://127.0.0.1:8000", "--no-autoupdate"],
                                  stdout=tunnel_log, stderr=subprocess.STDOUT)
try:
    deadline = time.monotonic() + 120
    PUBLIC_URL = None
    while time.monotonic() < deadline:
        if tunnel_process.poll() is not None:
            raise RuntimeError("トンネルの起動に失敗しました。")
        matches = re.findall(r"https://[a-z0-9-]+\.trycloudflare\.com", tunnel_path.read_text(errors="replace"))
        if matches:
            PUBLIC_URL = matches[-1]
            break
        time.sleep(1)
    if not PUBLIC_URL:
        raise TimeoutError("一時URLを取得できませんでした。")
    verified = False
    while time.monotonic() < deadline:
        try:
            response = requests.get(PUBLIC_URL + "/tagger-info", headers=headers, timeout=10)
            if response.status_code == 200 and response.json().get("pixai_model_name") == "v1.0":
                verified = True
                break
        except (requests.RequestException, ValueError):
            pass
        time.sleep(2)
    if not verified:
        raise RuntimeError("一時URLからAPIへの接続を確認できませんでした。")
    if requests.get(PUBLIC_URL + "/tagger-info", timeout=20).status_code != 401:
        raise RuntimeError("外部URLの認証チェックに失敗しました。")
    public_tag = requests.post(PUBLIC_URL + "/tag", headers=headers,
                               files={"file": ("test.png", sample.getvalue(), "image/png")}, timeout=120)
    public_tag.raise_for_status()
    assert isinstance(public_tag.json().get("tags"), list)
    print("Clientの「接続設定」に貼り付けてください。")
    print("接続先URL:", PUBLIC_URL)
    print("APIトークン:", API_TOKEN)
    print("HTTPSなのでポートは自動で443になります。")
except Exception:
    stop_services()
    print(tunnel_path.read_text(errors="replace")[-4000:])
    raise

## 任意：自分の画像で確認
下の `TEST_OWN_IMAGE` を `True` にした場合だけ、画像選択ダイアログを表示します。

In [ ]:
TEST_OWN_IMAGE = False
if TEST_OWN_IMAGE:
    from google.colab import files
    import mimetypes
    uploaded = files.upload()
    for name, content in uploaded.items():
        media_type = mimetypes.guess_type(name)[0] or "application/octet-stream"
        response = requests.post(PUBLIC_URL + "/tag", headers=headers,
                                 files={"file": (name, content, media_type)}, timeout=300)
        response.raise_for_status()
        print(name, response.json()["tags"])
else:
    print("画像選択は省略しました。Eagle Clientからタグ付けできます。")

## 終了
使い終わったらランタイムを削除してください。APIだけ停止する場合は、下の `STOP_NOW` を `True` にして実行します。
Colab内で変更した翻訳辞書や履歴は一時ファイルです。残したい場合はランタイム削除前にダウンロードしてください。

In [ ]:
STOP_NOW = False
if STOP_NOW:
    stop_services()
    print("APIとトンネルを停止しました。計算枠の消費を止めるにはランタイムも削除してください。")